# BUS1006 Applied Data Analytics
## Week 8: Group Project Starter — HDB Resale Data

**Group Name / Number:** Group 11  
**Team Members:**
1. JOHN EMMANUEL BRAGANZA MORILLO  (Student ID: 2501851)
2. JAREN LIM PIN-AN  (Student ID: 2501082)
3. TAN SI YAN  (Student ID: 2503097)
4. ANDREA GIAM ZHI YI  (Student ID: 2501524)
5. SEE JUN JIE MILLEN  (Student ID: 2503962)

**Date:** 2026-03-04

---

### 📌 Purpose of This Notebook
This is your **Week 8 milestone** for the group project.  
You are not submitting this for a grade — but completing it now means Week 9 will be smooth.

By the end of today's class, your group should have:
- [/] HDB data loaded and inspected
- [/] Agreed on your analysis date range (and why)
- [/] Identified all data quality issues
- [/] Made Drop vs Impute decisions for each column
- [/] Previewed the `remaining_lease` parsing challenge

---

### 📥 Before You Start: Download the Data

1. Go to **https://data.gov.sg**
2. Search for: `HDB Resale Flat Prices`
3. Download the CSV files (there may be multiple files for different time periods)
4. Place all CSV files in the **same folder** as this notebook

> 💡 **Expected size:** ~500MB total, ~970,000 rows across all files combined.

In [1]:
# Run this cell first — imports all libraries needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import glob
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (11, 5)
sns.set_style('whitegrid')

print("✅ Libraries loaded")

✅ Libraries loaded


---
## Task 1: Load the HDB Data

The HDB data may come as **multiple CSV files** (one per time period).  
We combine them using `pd.concat()`.

In [4]:
# ─────────────────────────────────────────────────────────────
# Task 1a: Load and combine all HDB CSV files
# ─────────────────────────────────────────────────────────────

# Option A — If you downloaded multiple CSV files:
# files = glob.glob('resale-flat-prices*.csv')
# print(f"Found {len(files)} CSV files: {files}")
# hdb = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# Option B — If you have one combined file:
# hdb = pd.read_csv('hdb_resale_combined.csv')

# ⬇️ UNCOMMENT whichever option applies to your downloaded files:

hdb = pd.read_csv('Resale flat prices based on registration date from Jan-2017 onwards.csv')

# After loading, print the shape
print(f"Total rows loaded: {len(hdb):,}")
print(f"Total columns:     {hdb.shape[1]}")

Total rows loaded: 225,902
Total columns:     11


---
## Task 2: Inspect the Data

Apply the same inspection steps we used in the class demonstration.

In [28]:
# Task 2a: Print shape, data types, and first 5 rows

print("📊 Data Overview:\n")
print(f"Shape:\n{hdb.shape}")
print("\nData types:")
print(hdb.dtypes)
print("\nFirst 5 rows:")
print(hdb.head())

📊 Data Overview:

Shape:
(225902, 11)

Data types:
month                  datetime64[ns]
town                           object
flat_type                      object
block                          object
street_name                    object
storey_range                   object
floor_area_sqm                float64
flat_model                     object
lease_commence_date             int64
remaining_lease                object
resale_price                  float64
dtype: object

First 5 rows:
       month        town flat_type block        street_name storey_range  \
0 2017-01-01  ANG MO KIO    2 ROOM   406  ANG MO KIO AVE 10     10 TO 12   
1 2017-01-01  ANG MO KIO    3 ROOM   108   ANG MO KIO AVE 4     01 TO 03   
2 2017-01-01  ANG MO KIO    3 ROOM   602   ANG MO KIO AVE 5     01 TO 03   
3 2017-01-01  ANG MO KIO    3 ROOM   465  ANG MO KIO AVE 10     04 TO 06   
4 2017-01-01  ANG MO KIO    3 ROOM   601   ANG MO KIO AVE 5     01 TO 03   

   floor_area_sqm      flat_model  lease_comm

In [14]:
# Task 2b: Print missing value count for each column
# Also print the % missing for any column that has missing data

missing_counts = hdb.isnull().sum()
missing_percent = (missing_counts / len(hdb)) * 100

print("📉 Missing Values:\n")
for col in hdb.columns:
    if missing_counts[col] > 0:
        print(f"{col}: {missing_counts[col]:,} missing ({missing_percent[col]:.2f}%)")
    else:
        print(f"{col.title()}: 0 missing")


📉 Missing Values:

Month: 0 missing
Town: 0 missing
Flat_Type: 0 missing
Block: 0 missing
Street_Name: 0 missing
Storey_Range: 0 missing
Floor_Area_Sqm: 0 missing
Flat_Model: 0 missing
Lease_Commence_Date: 0 missing
Remaining_Lease: 0 missing
Resale_Price: 0 missing


In [21]:
# Task 2c: Check for duplicate rows
# Print: number of duplicate rows found

duplicate_count = hdb.duplicated().sum()
print(f"🔁 Duplicate rows found: {duplicate_count:,}")


🔁 Duplicate rows found: 311


In [16]:
# Task 2d: Explore key columns
# Print unique values for: flat_type, storey_range
# Print the date range of the 'month' column

print("🔍 Exploring key columns:\n")
print("Unique values in 'flat_type':")
print(hdb['flat_type'].unique())
print("\nUnique values in 'storey_range':")
print(hdb['storey_range'].unique())
print(f"\nDate range in 'month' column:\n{hdb['month'].min()} to {hdb['month'].max()}")

🔍 Exploring key columns:

Unique values in 'flat_type':
['2 ROOM' '3 ROOM' '4 ROOM' '5 ROOM' 'EXECUTIVE' '1 ROOM'
 'MULTI-GENERATION']

Unique values in 'storey_range':
['10 TO 12' '01 TO 03' '04 TO 06' '07 TO 09' '13 TO 15' '19 TO 21'
 '22 TO 24' '16 TO 18' '34 TO 36' '28 TO 30' '37 TO 39' '49 TO 51'
 '25 TO 27' '40 TO 42' '31 TO 33' '46 TO 48' '43 TO 45']

Date range in 'month' column:
2017-01-01 00:00:00 to 2026-02-01 00:00:00


In [17]:
# Task 2e: Summary statistics for numeric columns
# Print .describe() for: resale_price, floor_area_sqm
# Look for: min/max outliers, unrealistic values

print("📈 Summary statistics for numeric columns:\n")
print(hdb[['resale_price', 'floor_area_sqm']].describe())

📈 Summary statistics for numeric columns:

       resale_price  floor_area_sqm
count  2.259020e+05   225902.000000
mean   5.267060e+05       96.748326
std    1.877142e+05       24.019711
min    1.400000e+05       31.000000
25%    3.880000e+05       81.000000
50%    4.950000e+05       93.000000
75%    6.300000e+05      112.000000
max    1.700000e+06      366.700000


---
## Task 3: Data Quality Report

Complete this section as a team. Double-click each markdown cell to edit it.

### 3a: Missing Values Decision Table

Fill in the table below based on your `.isnull().sum()` results:

| Column | Missing Count | Missing % | Decision | Reason |
|--------|--------------|-----------|----------|--------|
| remaining_lease | 709,050 | 72.94% | Impute with median | > 5% Threshold; Median is most robust and least influenced by outliers |

**Decision options:** Drop rows / Impute with mean / Impute with median / Impute with mode / Fill with 'Unknown'

### 3b: Date Range Decision

The full HDB dataset spans from 1990 to 2026 (~970,000 rows).  
Your group must decide on a date range to focus on.

**Your group's chosen date range:** 2017-01-01 to 2023-12-31

**Reason for this choice:**  
- *The analysis focuses on the period 2017–2023, a seven-year range selected to ensure consistency in the dataset structure and reduce data-cleaning complexity. Earlier datasets include different recording methods (approval date versus registration date), which may introduce inconsistencies during preprocessing. The selected period also captures a clear housing market cycle, including the stable resale market before COVID-19 and the sharp price increase observed after the pandemic. Limiting the analysis to seven recent years ensures computational efficiency while still providing a sufficiently large sample size for meaningful statistical and trend analysis.*

---

**Estimated rows after filtering:** *169,150*

In [23]:
# Task 3c: Filter data to your chosen date range
# Replace 'YYYY-MM' with your actual start and end dates

# First parse the month column:
# hdb['month'] = pd.to_datetime(hdb['month'])

# Then filter:
# START_DATE = 'YYYY-MM-DD'
# END_DATE   = 'YYYY-MM-DD'
# hdb_filtered = hdb[(hdb['month'] >= START_DATE) & (hdb['month'] <= END_DATE)]
# print(f"Rows after date filter: {len(hdb_filtered):,}")

hdb['month'] = pd.to_datetime(hdb['month'])

START_DATE = '2017-01-01'
END_DATE   = '2023-12-31'
hdb_filtered = hdb[(hdb['month'] >= START_DATE) & (hdb['month'] <= END_DATE)]
print(f"Rows after date filter: {len(hdb_filtered):,}")

Rows after date filter: 169,150


---
## Task 4: Preview the remaining_lease Challenge

The `remaining_lease` column comes as text like `"63 years 01 month"`.  
You must convert it to a number to answer **Business Question 2** (lease depreciation).

You will do the full conversion in **Week 9**.  
For now, inspect the column and understand the problem.

In [24]:
# Task 4a: Inspect the remaining_lease column
# Print: data type, 10 sample values, any rows where it is missing

print("🔍 Inspecting 'remaining_lease' column:\n")
print(f"Data type: {hdb_filtered['remaining_lease'].dtype}")
print("\n10 sample values:")
print(hdb_filtered['remaining_lease'].sample(10))
print(f"\nMissing values: {hdb_filtered['remaining_lease'].isnull().sum()}")

🔍 Inspecting 'remaining_lease' column:

Data type: object

10 sample values:
104217    62 years 06 months
5802      78 years 09 months
110301    78 years 11 months
168124              64 years
112588    92 years 04 months
160704    90 years 05 months
141937    63 years 07 months
137588    63 years 03 months
137738              65 years
95417     62 years 11 months
Name: remaining_lease, dtype: object

Missing values: 0


In [25]:
# Task 4b: Test the parse_lease function on sample values
# This is a PREVIEW — full implementation in Week 9

def parse_lease_years(text):
    """
    Converts remaining_lease text to numeric years.
    Examples:
      '63 years 01 month'  → 63.08
      '45 years'           → 45.0
      '70 years 06 months' → 70.5
    """
    if pd.isna(text):
        return np.nan
    text = str(text).strip()
    years = 0
    months = 0
    if 'year' in text:
        years = int(text.split('year')[0].strip())
    if 'month' in text:
        months = int(text.split('month')[0].split()[-1].strip())
    return round(years + months / 12, 2)

# Test on sample values:
test_cases = ['63 years 01 month', '45 years', '70 years 06 months', '99 years', None]
print("Test results:")
for t in test_cases:
    print(f"  Input: {str(t):30s} → Output: {parse_lease_years(t)}")

# PREVIEW: Apply to your actual data:
# hdb_filtered['remaining_lease_years'] = hdb_filtered['remaining_lease'].apply(parse_lease_years)
# print(hdb_filtered['remaining_lease_years'].describe())
hdb_filtered['remaining_lease_years'] = hdb_filtered['remaining_lease'].apply(parse_lease_years)
print(hdb_filtered['remaining_lease_years'].describe())

Test results:
  Input: 63 years 01 month              → Output: 63.08
  Input: 45 years                       → Output: 45.0
  Input: 70 years 06 months             → Output: 70.5
  Input: 99 years                       → Output: 99.0
  Input: None                           → Output: nan
count    169150.000000
mean         74.682456
std          13.856807
min          42.080000
25%          63.420000
50%          74.670000
75%          88.000000
max          97.750000
Name: remaining_lease_years, dtype: float64


---
## Task 5: Feature Engineering Preview

Your group project requires creating these calculated columns.  
Preview them now, implement fully in Week 9.

In [26]:
# Task 5a: Create price_per_sqm column
# Formula: resale_price / floor_area_sqm
# This is needed for Business Question 1 (best value towns)

# hdb_filtered['price_per_sqm'] = hdb_filtered['resale_price'] / hdb_filtered['floor_area_sqm']
# print("price_per_sqm stats:")
# print(hdb_filtered['price_per_sqm'].describe().round(0))

# YOUR CODE HERE (uncomment and run once hdb_filtered exists)
hdb_filtered['price_per_sqm'] = hdb_filtered['resale_price'] / hdb_filtered['floor_area_sqm']
print("price_per_sqm stats:")
print(hdb_filtered['price_per_sqm'].describe().round(0))


price_per_sqm stats:
count    169150.0
mean       5092.0
std        1413.0
min        2090.0
25%        4118.0
50%        4815.0
75%        5698.0
max       15171.0
Name: price_per_sqm, dtype: float64


In [27]:
# Task 5b: Map towns to regions
# This is needed for Business Question 4 (geographic differences)
# You must build this dictionary based on your knowledge of Singapore geography

# This is a STARTER — your team must verify and complete this mapping
town_to_region = {
    # CENTRAL
    'BISHAN': 'Central', 'BUKIT MERAH': 'Central', 'BUKIT TIMAH': 'Central',
    'CENTRAL AREA': 'Central', 'GEYLANG': 'Central', 'KALLANG/WHAMPOA': 'Central',
    'MARINE PARADE': 'Central', 'QUEENSTOWN': 'Central', 'TOA PAYOH': 'Central',
    # EAST
    'BEDOK': 'East', 'PASIR RIS': 'East', 'TAMPINES': 'East',
    # NORTH
    'SEMBAWANG': 'North', 'WOODLANDS': 'North', 'YISHUN': 'North',
    # NORTH-EAST
    'ANG MO KIO': 'North-East', 'HOUGANG': 'North-East', 'PUNGGOL': 'North-East',
    'SENGKANG': 'North-East', 'SERANGOON': 'North-East',
    # WEST
    'BUKIT BATOK': 'West', 'BUKIT PANJANG': 'West', 'CHOA CHU KANG': 'West',
    'CLEMENTI': 'West', 'JURONG EAST': 'West', 'JURONG WEST': 'West',
}

# Apply the mapping:
# hdb_filtered['region'] = hdb_filtered['town'].map(town_to_region)
# unmapped = hdb_filtered[hdb_filtered['region'].isna()]['town'].unique()
# if len(unmapped) > 0:
#     print(f"⚠️  Towns not yet mapped: {unmapped}")
#     print("Add these to town_to_region before continuing")
# else:
#     print("✅ All towns mapped to regions")
hdb_filtered['region'] = hdb_filtered['town'].map(town_to_region)
unmapped = hdb_filtered[hdb_filtered['region'].isna()]['town'].unique()
if len(unmapped) > 0:
    print(f"⚠️  Towns not yet mapped: {unmapped}")
    print("Add these to town_to_region before continuing")
else:
    print("✅ All towns mapped to regions")


print("Region mapping ready.")
print(f"Towns mapped: {len(town_to_region)}")
print("\n⚠️  Check: run print(hdb_filtered['town'].unique()) and verify all towns are in this dict")

✅ All towns mapped to regions
Region mapping ready.
Towns mapped: 26

⚠️  Check: run print(hdb_filtered['town'].unique()) and verify all towns are in this dict


---
## ✅ Week 8 Group Milestone Checklist

Before you leave today, confirm your group has completed:

| Task | Done? | Notes |
|------|-------|-------|
| Data downloaded from data.gov.sg | ☑ | |
| Data loaded into notebook | ☑ | |
| `.shape` confirmed (~970K rows) | ☑ | |
| `.info()` and `.dtypes` checked | ☑ | |
| Missing values per column identified | ☑ | |
| Duplicate row count checked | ☑ | |
| Drop/Impute table completed above | ☑ | |
| Date range agreed + reason written | ☑ | From: 2017 To: 2023 |
| `parse_lease_years()` tested | ☑ | |
| `town_to_region` mapping verified | ☑ | |

---

### 📅 What's Next: Week 9 Preview

In Week 9 you will:
1. Apply the full cleaning pipeline (duplicates → missing values → data types)
2. Parse `remaining_lease` to numeric years using `parse_lease_years()`
3. Remove outliers (IQR method or domain rules)
4. Create all required features: `price_per_sqm`, `remaining_lease_years`, `region`, `year`
5. Produce a clean, analysis-ready dataset

**Come to Week 9 with your data loaded and this notebook complete.**

---
*BUS1006 Applied Data Analytics — Singapore Institute of Technology*